In [6]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report

# 1. PATH FIXING
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. IMPORTS (Updated with your actual class name)
from backend.core_ai.models.fusion_net import DeepGuardFusionModel
# Agar aapki class ka naam DeepGuardDataset hai:
from datasets.loaders.multi_modal_loader import DeepGuardDataset 

# 3. SETUP
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "../saved_models/production/deepguard_fusion_v1.pth"
TEST_DATA_DIR = "E:/Deepfake_Dataset/test" # 👈 Apni E: Drive ka sahi path check karein

# 4. LOAD MODEL (FP16 Mode for RAM Safety)
model = DeepGuardFusionModel().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.half() # 👈 RAM bachane ke liye model ko aadha (FP16) kar diya
    model.eval()
    print("🚀 DeepGuard Engine Ready & Weights Loaded!")

# 5. EVALUATION LOOP
def evaluate_performance(num_videos=40):
    # Dataset Load karein
    test_ds = DeepGuardDataset(TEST_DATA_DIR)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=True)
    
    y_true, y_pred = [], []
    
    print(f"🔍 Scanning {num_videos} videos from E: Drive...")
    with torch.no_grad():
        for i, (video, flow, fft, audio, label) in enumerate(test_loader):
            if i >= num_videos: break
            
            # Model ke hisab se data bhejna
            video = video.to(DEVICE).half() # Half precision
            flow, fft, audio = flow.to(DEVICE).half(), fft.to(DEVICE).half(), audio.to(DEVICE).half()
            
            output = model(video, flow, fft, audio)
            prob = torch.sigmoid(output).item()
            
            y_true.append(label.item())
            y_pred.append(1 if prob >= 0.5 else 0)
            
            if i % 5 == 0: print(f"Processing... {i}/{num_videos}")

    return y_true, y_pred

# --- EXECUTION & GRAPHS ---
y_true, y_pred = evaluate_performance(40)

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title('DeepGuard Final Evaluation')
plt.show()

print(classification_report(y_true, y_pred))

: 